# Lab 20-05 · Sampling Noise, Switch Distortion, Jitter and Latch Regeneration

**Predict first**, then run. The handout is [README.md](README.md). The devices are the chapter's illustrative T130 NMOS and PMOS.

In [1]:
import collections
import math
import re
import shutil
import subprocess
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import sh

C = sh.ktc_cap(12, 1.6)
TAU = 0.5 / 100e6 / math.log(2 ** 13)
W = sh.L_MIN / (sh.MU_N * sh.COX * (TAU / C) * (sh.VDD - sh.vth(1.0)))
print(f"C = {C * 1e12:.3f} pF per side, R_on target = {TAU / C:.0f} ohm, W = {W * 1e6:.3f} um")


def ngspice(netlist):
    if shutil.which("ngspice") is None:
        print(f"ngspice is not installed: skipping {netlist} (the README lists the expected values)")
        return collections.defaultdict(lambda: float("nan"))
    out = subprocess.run(["ngspice", "-b", netlist], capture_output=True, text=True).stdout
    return {m.group(1): float(m.group(2)) for m in re.finditer(r"^(\w+)\s*=\s*([-+0-9.eE]+)", out, re.M)}

C = 0.651 pF per side, R_on target = 852 ohm, W = 0.888 um


## Part 1 · kT/C

*Predict* the integrated noise on 0.651 pF through 100 ohm and through 10 kohm.

In [2]:
print(f"sqrt(kT/C) = {math.sqrt(sh.KT / C) * 1e6:.2f} uV; at 1 pF {math.sqrt(sh.KT / 1e-12) * 1e6:.2f} uV")
print(ngspice("ktc.cir"))

sqrt(kT/C) = 79.74 uV; at 1 pF 64.36 uV
{'na': 7.972626e-05, 'nb': 7.97324e-05, 'nc': 6.435478e-05}


## Part 2 · On-resistance

*Predict* where the plain switch turns off.

In [3]:
plt.figure()
vin = np.linspace(0.05, 1.15, 221)
plain = [sh.ron(W, sh.VDD, x) for x in vin]
boot = [sh.ron(W, x + sh.VDD, x) for x in vin]
plt.semilogy(vin, plain, label="gate at V_DD"); plt.semilogy(vin, boot, label="bootstrapped")
plt.axhline(TAU / C, color="k", lw=0.8); plt.ylim(300, 3e4); plt.xlabel("input (V)"); plt.ylabel("R_on (ohm)"); plt.legend(); plt.show()
for x in (0.2, 0.6, 1.0):
    print(f"{x} V: plain {sh.ron(W, sh.VDD, x):.0f} ohm, bootstrapped {sh.ron(W, x + sh.VDD, x):.0f} ohm")

0.2 V: plain 958 ohm, bootstrapped 687 ohm
0.6 V: plain 14106 ohm, bootstrapped 767 ohm
1.0 V: plain inf ohm, bootstrapped 852 ohm


## Part 3 · Tracking distortion

*Predict* HD2 and HD3 at 49 MHz from the chapter's first-order estimates, with the body grounded and tied to the source.

In [4]:
for mode in ("boot", "boot_nobody"):
    for f in (10e6, 49e6):
        r = sh.track(W, C, f, 0.4, mode=mode)
        print(f"{mode:12s} {f / 1e6:4.0f} MHz: HD2 {r['hd2_db']:7.1f} dBc, HD3 {r['hd3_db']:7.1f} dBc, gain {r['gain']:.4f}")
print(ngspice("switch.cir"))

boot           10 MHz: HD2   -55.5 dBc, HD3   -87.9 dBc, gain 0.9995
boot           49 MHz: HD2   -41.5 dBc, HD3   -64.4 dBc, gain 0.9887


boot_nobody    10 MHz: HD2   -80.6 dBc, HD3  -123.6 dBc, gain 0.9996
boot_nobody    49 MHz: HD2   -53.2 dBc, HD3   -83.1 dBc, gain 0.9916


{'n': 16385.0, 'hd2a': -41.5486, 'hd3a': -64.4088, 'hd3b': -83.0952}


## Part 4 · SNR budget

*Predict* the input frequency at which 1 ps of jitter costs 3 dB.

In [5]:
plt.figure()
f = np.logspace(6, 9.3, 300)
for s in (0.1e-12, 0.3e-12, 1e-12):
    plt.semilogx(f, [sh.snr_db(12, 1.6, C, s, x) for x in f], label=f"{s * 1e12:g} ps")
plt.xlabel("input frequency (Hz)"); plt.ylabel("SNR (dB)"); plt.legend(); plt.show()
floor = sh.snr_db(12, 1.6, C)
knee = math.sqrt(10 ** (-floor / 10)) / (2 * math.pi * 1e-12)
print(f"floor {floor:.2f} dB; 3 dB knee for 1 ps at {knee / 1e6:.1f} MHz")

floor 71.00 dB; 3 dB knee for 1 ps at 44.9 MHz


## Part 5 · Regeneration

*Predict* the decision time for half an LSB from tau ln(V_L/v_0), with tau = 20 ps and A_int = 6.75.

In [6]:
lsb = 1.6 / 4096
for vi in (lsb / 2, 1e-3, 10e-3):
    v0 = 6.75 * vi
    est = sh.decision_time(vi, 6.75, 20e-12)
    sim = sh.regen(v0, 3e-3, 60e-15, 0.3e-3, 0.6)
    print(f"v_in = {vi * 1e3:7.4f} mV: estimate {est * 1e12:6.1f} ps, current-limited simulation {sim * 1e12:6.1f} ps")
print(ngspice("latch.cir"))
p = sh.sar_meta_probability(300e-12, 6.75, 20e-12, 12, 1.6)
print(f"SAR metastability probability with 300 ps per decision: {p:.3g} per conversion")

v_in =  0.1953 mV: estimate  122.4 ps, current-limited simulation  146.5 ps
v_in =  1.0000 mV: estimate   89.7 ps, current-limited simulation  113.9 ps
v_in = 10.0000 mV: estimate   43.7 ps, current-limited simulation   67.4 ps
{'ta': 1.465258e-10, 'tb': 1.13859e-10, 'tc': 6.743291e-11}
SAR metastability probability with 300 ps per decision: 0.000139 per conversion
